In [ ]:
import torch
import torch.nn as nn
from car_env import CarEnv

In [ ]:
class CarMLP(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.Linear(32, 64),
            nn.Linear(64, output_dim),
            nn.Tanh()
        )
        self.input_dim = input_dim
        self.output_dim = output_dim
    def forward(self, x):
        return self.net(x)

In [ ]:
class CarCNN(nn.Module):
    def __init__(self, w_h, c, c_1, c_2, output_dim, kernel_size=3, padding = 0, stride = 2):
        output_h_w = ((((int(((w_h - kernel_size + 2 * padding) / stride) + 1)) - kernel_size + 2 * padding) / stride) + 1)
        self.net = nn.Sequential(
            nn.Conv2d(c, c_1, kernel_size, stride=stride, padding=padding),
            nn.GELU(),
            nn.Conv2d(c_1, c_2, kernel_size, stride=stride, padding=padding),
            nn.GELU(),
            nn.Fllaten(),
            nn.Linear(c_2 * output_h_w * output_h_w, 128),
            nn.LeakyReLU(),
            nn.Linear(128, output_dim),
            nn.Tanh()
        )
        self.w_h = w_h
        self.c = c
        self.c_1 = c_1
        self.c_2 = c_2
        self.output_dim = output_dim
        self.kernel_size = kernel_size
        self.padding = padding
        self.stride = stride
    
    def forward(self, x):
        return self.net(x)

In [ ]:
class CarLSTM(nn.Module):
    def __init__(self, w_h, c, c_1, c_2, output_dim, hidden_size, num_layers=2, kernel_size=3, padding = 0, stride = 2):
        self.w_h = w_h
        self.c = c
        self.c_1 = c_1
        self.c_2 = c_2
        self.output_dim = output_dim
        self.kernel_size = kernel_size
        self.padding = padding
        self.stride = stride
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        output_h_w = ((((int(((w_h - kernel_size + 2 * padding) / stride) + 1)) - kernel_size + 2 * padding) / stride) + 1)
        self.conv_net = nn.Sequential(
            nn.Conv2d(c, c_1, kernel_size, stride=stride, padding=padding),
            nn.GELU(),
            nn.Conv2d(c_1, c_2, kernel_size, stride=stride, padding=padding),
            nn.GELU(),
            nn.Fllaten(),
            nn.Linear(c_2 * output_h_w * output_h_w, 128),
            nn.LeakyReLU(),
            nn.Linear(128, hidden_size)
        )
        
        self.lstm_net = nn.Sequential(
            nn.LSTM(hidden_size, hidden_size, num_layers=num_layers, batch_first=True)
        )

        self.mlp = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.LeakyReLU(),
            nn.Linear(64, output_dim),
            nn.Tanh()
        )

    def forward(self, x):
        # x => b, seq_length, c, h, w
        batch_size, seq_length, c, h, w = x.shape
        
        pass

In [ ]:
from car_env import CarEnv
env = CarEnv(n_cars=50, difficulty=0.7, seed=0, render_mode=None)  # "human" ca să vezi fereastra
obs = env.reset()
while not env.all_done:
    actions = 2 * torch.rand((50, 2)) - 1           # (n_cars, 2): [virare, accelerație], ambele în [-1, 1]
    obs, rewards, dones, info = env.step(actions)
fitness = env.fitness                  # cât a parcurs fiecare mașină, în ture (1.0 = o tură)